# B5: Leaf-JEPA Linear Probe

**Leaf-JEPA IRP** | Stage 3 — Baseline Establishment

## Rationale

B5 is the other half of the **RQ1 central comparison**. Identical to B3 in every way —
same architecture, same frozen encoder, same linear head, same hyperparameters, same
evaluation — except the encoder weights come from **Stage 4 domain pretraining** instead
of Meta's ImageNet checkpoint.

**B5 - B3 = pure representation quality gain from domain pretraining.**

This is the cleanest possible comparison. No confounders exist.

**Prerequisite**: Stage 4 must be complete and LEAF_JEPA_CHECKPOINT must be set in config.

## Initialization

In [1]:
# Setup and checkpoint verification
import sys, json, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.amp import autocast
from torch.utils.data import DataLoader, TensorDataset

sys.path.insert(0, str(Path("../..").resolve()))
sys.path.insert(0, str(Path("..").resolve()))

from stage3_baseline_establishment.config_stage3 import *
from stage3_baseline_establishment.baseline_utils import (
    seed_everything, PlantVillageDataset, load_split,
    save_results, plot_confusion_matrix, plot_tsne, load_ijepa_encoder
)

from stage2_dataset_preparation.outputs.augmentation.transforms import (
    get_pretrain_transform, get_eval_transform, get_finetune_transform
)

verify_config()
seed_everything(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── CRITICAL: Check Leaf-JEPA checkpoint exists ──
if LEAF_JEPA_CHECKPOINT is None or not Path(LEAF_JEPA_CHECKPOINT).exists():
    print("=" * 60)
    print("  LEAF-JEPA CHECKPOINT NOT FOUND")
    print("=" * 60)
    print(f"  LEAF_JEPA_CHECKPOINT = {LEAF_JEPA_CHECKPOINT}")
    print("  Complete Stage 4 first, then update config_stage3.py.")
    print("  Exiting B5 gracefully.")
    raise SystemExit("Stage 4 checkpoint required for B5.")

print(f"  Leaf-JEPA checkpoint: {LEAF_JEPA_CHECKPOINT}")

hyperparams = {
    "baseline": "B5",
    "model": "Leaf-JEPA ViT-H/14 Linear Probe (domain-pretrained)",
    "encoder_params": "632M (ALL FROZEN)",
    "head_params": "~49K",
    "checkpoint": str(LEAF_JEPA_CHECKPOINT),
    "optimiser": "AdamW (head only)",
    "lr": LP_LR,
    "batch_size": LP_BATCH_SIZE,
    "max_epochs": LP_EPOCHS,
    "patience": LP_PATIENCE,
    "note": "IDENTICAL setup to B3 — only encoder weights differ",
}
BASELINES_DIR.mkdir(parents=True, exist_ok=True)
save_results(hyperparams, BASELINES_DIR / "B5_hyperparams.json")

  ✅  All config checks passed.
  LEAF-JEPA CHECKPOINT NOT FOUND
  LEAF_JEPA_CHECKPOINT = None
  Complete Stage 4 first, then update config_stage3.py.
  Exiting B5 gracefully.


SystemExit: Stage 4 checkpoint required for B5.

D:\IRP\leaf-jepa\.venv\lib\site-packages\IPython\core\interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## Extract and cache features

In [ ]:
print("Loading Leaf-JEPA encoder...")
encoder = load_ijepa_encoder(LEAF_JEPA_CHECKPOINT, VIT_MODEL_NAME, device)

for param in encoder.parameters():
    param.requires_grad = False
encoder.eval()

n_frozen = sum(p.numel() for p in encoder.parameters())
print(f"  Encoder params: {n_frozen:,} (all frozen)")

train_paths, train_labels, class_names = load_split(SPLITS_DIR / "train_split.json")
val_paths, val_labels, _ = load_split(SPLITS_DIR / "val_split.json")
test_paths, test_labels, _ = load_split(SPLITS_DIR / "test_split.json")

eval_tf = get_eval_transform()

def extract_features(paths, labels, desc="Extracting"):
    ds = PlantVillageDataset(paths, labels, transform=eval_tf)
    loader = DataLoader(ds, batch_size=LP_BATCH_SIZE, shuffle=False,
                        num_workers=4, pin_memory=True)
    all_feats, all_labs = [], []
    print(f"  {desc}: {len(ds)} images...")
    with torch.no_grad():
        for images, labs in loader:
            images = images.to(device, non_blocking=True)
            with autocast(device_type="cuda", enabled=True):
                feats = encoder(images)
            all_feats.append(feats.float().cpu())
            all_labs.append(labs)
    return torch.cat(all_feats), torch.cat(all_labs)

train_feats, train_labs = extract_features(train_paths, train_labels, "Train")
val_feats, val_labs = extract_features(val_paths, val_labels, "Val")
test_feats, test_labs = extract_features(test_paths, test_labels, "Test")

del encoder
torch.cuda.empty_cache()
print(f"  Feature dim: {train_feats.shape[1]}")

## Train linear head

In [ ]:
import wandb
from sklearn.metrics import f1_score as sk_f1, accuracy_score as sk_acc, confusion_matrix as sk_cm
from baseline_utils import EarlyStopping

train_floader = DataLoader(TensorDataset(train_feats, train_labs),
                           batch_size=LP_BATCH_SIZE, shuffle=True)
val_floader   = DataLoader(TensorDataset(val_feats, val_labs),
                           batch_size=LP_BATCH_SIZE, shuffle=False)
test_floader  = DataLoader(TensorDataset(test_feats, test_labs),
                           batch_size=LP_BATCH_SIZE, shuffle=False)

head = nn.Linear(EMBED_DIM, NUM_CLASSES).to(device)
nn.init.xavier_uniform_(head.weight)
nn.init.zeros_(head.bias)

criterion = nn.CrossEntropyLoss()
optimiser = torch.optim.AdamW(head.parameters(), lr=LP_LR, weight_decay=LP_WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=LP_EPOCHS)
es = EarlyStopping(patience=LP_PATIENCE)

if WANDB_ENTITY:
    os.environ["WANDB__SERVICE_WAIT"] = "10"
    os.environ["WANDB_DISABLED"] = "true"
    try:
        wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                   name="B5-LeafJEPA-LinearProbe", reinit=True, config=hyperparams)
    except Exception:
        print("WandB init failed — training without logging")
        WANDB_ENTITY = False

print("\nTraining B5 linear probe on Leaf-JEPA features...")
t0 = time.time()
for epoch in range(LP_EPOCHS):
    head.train()
    for feats_b, labs_b in train_floader:
        feats_b, labs_b = feats_b.to(device), labs_b.to(device)
        loss = criterion(head(feats_b), labs_b)
        optimiser.zero_grad(); loss.backward(); optimiser.step()
    scheduler.step()

    head.eval()
    vp, vt = [], []
    with torch.no_grad():
        for fb, lb in val_floader:
            vp.extend(head(fb.to(device)).argmax(1).cpu().numpy())
            vt.extend(lb.numpy())
    val_f1 = sk_f1(vt, vp, average="macro", zero_division=0)

    if WANDB_ENTITY:
        wandb.log({"val_macro_f1": val_f1, "epoch": epoch})

    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1:3d} | val_F1={val_f1:.4f}")

    if val_f1 > es.best_score + es.min_delta:
        es.best_score = val_f1
        es.best_state = {k: v.clone() for k, v in head.state_dict().items()}
        es.counter = 0
    else:
        es.counter += 1
        if es.counter >= es.patience:
            print(f"  Early stop at epoch {epoch+1}")
            break

elapsed = time.time() - t0
if es.best_state:
    head.load_state_dict(es.best_state)

# Test
head.eval()
tp, tt = [], []
with torch.no_grad():
    for fb, lb in test_floader:
        tp.extend(head(fb.to(device)).argmax(1).cpu().numpy())
        tt.extend(lb.numpy())

test_f1 = sk_f1(tt, tp, average="macro", zero_division=0)
test_acc = sk_acc(tt, tp)
per_class = sk_f1(tt, tp, average=None, zero_division=0)
cm = sk_cm(tt, tp, labels=list(range(NUM_CLASSES)))

print(f"\n  B5 Test macro-F1:  {test_f1:.4f}")
print(f"  B5 Test accuracy:  {test_acc:.4f}")

# ── RQ1 RESULT ──
b3_path = BASELINES_DIR / "B3_aggregate.json"
if b3_path.exists():
    with open(b3_path) as f:
        b3 = json.load(f)
    gap = test_f1 - b3["macro_f1"]
    print(f"\n  *** RQ1 RESULT ***")
    print(f"  B3 (generic):  {b3['macro_f1']:.4f}")
    print(f"  B5 (domain):   {test_f1:.4f}")
    print(f"  Gap (B5 - B3): {gap:+.4f} ({gap*100:+.1f} percentage points)")

    rq1_summary = {"B3_macro_f1": b3["macro_f1"], "B5_macro_f1": float(test_f1),
                    "gap": float(gap), "gap_pct_points": float(gap * 100)}
    save_results(rq1_summary, BASELINES_DIR / "RQ1_summary.json")

if WANDB_ENTITY:
    wandb.log({"test_macro_f1": test_f1, "test_accuracy": test_acc})
    wandb.finish()

b5_aggregate = {
    "baseline": "B5", "model": "Leaf-JEPA ViT-H/14 Linear Probe (domain)",
    "macro_f1": float(test_f1), "accuracy": float(test_acc),
    "per_class_f1": [float(f) for f in per_class],
    "training_time_s": elapsed,
    "best_val_f1": float(es.best_score),
    "trainable_params": sum(p.numel() for p in head.parameters()),
    "encoder_params": n_frozen,
}
save_results(b5_aggregate, BASELINES_DIR / "B5_aggregate.json")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
plot_confusion_matrix(cm, class_names,
                      FIGURES_DIR / "B5_confusion_matrix.png",
                      title=f"B5 Leaf-JEPA LP | F1={test_f1:.3f}")

## t-SNE visualisation + comparison with B3

In [ ]:

test_feats_np = test_feats.numpy()
test_labs_np = np.array(test_labels)

plot_tsne(test_feats_np, test_labs_np, class_names,
          FIGURES_DIR / "B5_tsne_leafjepa.png",
          title="B5: Leaf-JEPA Feature Space (t-SNE)")

print("\nB5 complete.")
print("Compare B3 vs B5 t-SNE figures side-by-side for the dissertation.")

## Critical Analysis

B5 completes the RQ1 comparison:

1. **B5 - B3 gap** — This is the headline result. A positive gap confirms domain pretraining adds value. Document the exact numbers with confidence intervals.

2. **t-SNE comparison** — If B5 shows tighter, more disease-separated clusters than B3, this is qualitative evidence that domain pretraining learns disease-specific features rather than generic visual features.

3. **Per-class improvement** — Check which classes improved most from B3 to B5. If classes with high background composition (Stage 2 finding) improved most, this supports the disease-region-biased masking contribution.

4. **If gap is small** — This is still a valid finding. Discuss why: PlantVillage may be visually simple enough that generic features suffice, or the pretraining data volume (~54K) may be insufficient.